# 11 — Market-anchored correction model and edge selector

This notebook trains a small correction around market probabilities instead of trying to replace the market entirely.

Concept:

```text
market_T24 probability = baseline
football model probability = independent model view
score model probability = scoreline-based model view
correction model = small adjustment around market_T24
```

Why this design:
- the market is already a strong probability baseline;
- a large model can overfit on a small odds dataset;
- the goal is calibrated forecasting, not a black-box replacement for the market.

API calls: 0.
Real-money stake: 0.

Input:
- `data/processed/07_walkforward_market_backtest/walkforward_eval_dataset.csv`
- `data/processed/10_outcome_score_market_features/outcome_model_predictions.csv`
- `data/processed/10_outcome_score_market_features/score_model_predictions.csv`

Output:
- `data/processed/11_market_anchored_correction_report_bundle.zip`


In [ ]:
# Cell 1. Imports and helpers.

from pathlib import Path
import json
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"

IN_07_DIR = PROCESSED_DIR / "07_walkforward_market_backtest"
IN_10_DIR = PROCESSED_DIR / "10_outcome_score_market_features"

OUT_DIR = PROCESSED_DIR / "11_market_anchored_correction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    return path

def normalize_rows(arr):
    arr = np.asarray(arr, dtype=float)
    arr = np.clip(arr, 1e-6, 1.0)
    return arr / arr.sum(axis=1, keepdims=True)

def multiclass_brier(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    one_hot = np.zeros_like(proba, dtype=float)
    one_hot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((proba - one_hot) ** 2, axis=1)))

def evaluate_probs(name, y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    proba = normalize_rows(proba)
    pred = np.argmax(proba, axis=1)

    return {
        "model": name,
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "log_loss": float(log_loss(y_true, proba, labels=[0, 1, 2])),
        "brier": multiclass_brier(y_true, proba),
    }

def profit_from_pick(row):
    if int(row["selection_won"]) == 1:
        return float(row["best_odds_24h"]) - 1.0
    return -1.0

def safe_div(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b

def bootstrap_ci(values, n_boot=3000, seed=SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}

    rng = np.random.default_rng(seed)
    means = []

    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        means.append(np.mean(sample))

    return {
        "mean": float(np.mean(values)),
        "ci_low": float(np.quantile(means, 0.025)),
        "ci_high": float(np.quantile(means, 0.975)),
    }

print("Setup OK.")


In [ ]:
# Cell 2. Load inputs.

eval_path = IN_07_DIR / "walkforward_eval_dataset.csv"
outcome_pred_path = IN_10_DIR / "outcome_model_predictions.csv"
score_pred_path = IN_10_DIR / "score_model_predictions.csv"

if not eval_path.exists():
    raise FileNotFoundError(
        "Missing 07 walkforward_eval_dataset.csv. Run notebook 07 first."
    )

eval_df = pd.read_csv(eval_path)

if outcome_pred_path.exists():
    outcome_preds = pd.read_csv(outcome_pred_path)
else:
    outcome_preds = pd.DataFrame()
    print("WARNING: outcome_model_predictions.csv not found.")

if score_pred_path.exists():
    score_preds = pd.read_csv(score_pred_path)
else:
    score_preds = pd.DataFrame()
    print("WARNING: score_model_predictions.csv not found.")

for df in [eval_df, outcome_preds, score_preds]:
    if len(df) > 0 and "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

eval_df = eval_df.dropna(subset=["date", "outcome"]).copy()
eval_df["outcome"] = eval_df["outcome"].astype(int)

print("eval_df:", eval_df.shape)
print("outcome_preds:", outcome_preds.shape)
print("score_preds:", score_preds.shape)

display(eval_df.head())


In [ ]:
# Cell 3. Build base probability table.

base = eval_df[[
    "date",
    "home_team",
    "away_team",
    "outcome",
    "T_minus_24h_market_p_away_devig",
    "T_minus_24h_market_p_draw_devig",
    "T_minus_24h_market_p_home_devig",
    "T_minus_24h_best_away_odds",
    "T_minus_24h_best_draw_odds",
    "T_minus_24h_best_home_odds",
    "T_minus_1h_best_away_odds",
    "T_minus_1h_best_draw_odds",
    "T_minus_1h_best_home_odds",
    "football_p_away",
    "football_p_draw",
    "football_p_home",
]].copy()

base = base.rename(columns={
    "T_minus_24h_market_p_away_devig": "market_p_away",
    "T_minus_24h_market_p_draw_devig": "market_p_draw",
    "T_minus_24h_market_p_home_devig": "market_p_home",
    "T_minus_24h_best_away_odds": "best_away_odds_24h",
    "T_minus_24h_best_draw_odds": "best_draw_odds_24h",
    "T_minus_24h_best_home_odds": "best_home_odds_24h",
    "T_minus_1h_best_away_odds": "best_away_odds_1h",
    "T_minus_1h_best_draw_odds": "best_draw_odds_1h",
    "T_minus_1h_best_home_odds": "best_home_odds_1h",
})

# Add outcome-model prediction from 10 if available.
if len(outcome_preds) > 0:
    # Prefer honest T24 model if present.
    preferred_models = [
        "football_plus_market_T24",
        "market_T24_only",
        "football_only",
    ]

    chosen_outcome = pd.DataFrame()

    for model_name in preferred_models:
        part = outcome_preds[outcome_preds["model"] == model_name].copy()
        if len(part) > 0:
            chosen_outcome = part
            print("Chosen outcome pred model:", model_name)
            break

    if len(chosen_outcome) > 0:
        chosen_outcome = chosen_outcome[[
            "date",
            "home_team",
            "away_team",
            "p_away",
            "p_draw",
            "p_home",
        ]].rename(columns={
            "p_away": "outcome_model_p_away",
            "p_draw": "outcome_model_p_draw",
            "p_home": "outcome_model_p_home",
        })

        base = base.merge(
            chosen_outcome,
            on=["date", "home_team", "away_team"],
            how="left",
        )

# Add score-derived probabilities from 10 if available.
if len(score_preds) > 0:
    preferred_score_models = [
        "score_football_plus_market_T24",
        "score_football_only",
    ]

    chosen_score = pd.DataFrame()

    for model_name in preferred_score_models:
        part = score_preds[score_preds["model"] == model_name].copy()
        if len(part) > 0:
            chosen_score = part
            print("Chosen score pred model:", model_name)
            break

    if len(chosen_score) > 0:
        chosen_score = chosen_score[[
            "date",
            "home_team",
            "away_team",
            "p_away_from_score",
            "p_draw_from_score",
            "p_home_from_score",
            "lambda_home",
            "lambda_away",
        ]].rename(columns={
            "p_away_from_score": "score_p_away",
            "p_draw_from_score": "score_p_draw",
            "p_home_from_score": "score_p_home",
        })

        base = base.merge(
            chosen_score,
            on=["date", "home_team", "away_team"],
            how="left",
        )

base = base.sort_values("date").reset_index(drop=True)

base.to_csv(
    OUT_DIR / "base_probability_table.csv",
    index=False,
)

print("base:", base.shape)
display(base.head())


In [ ]:
# Cell 4. Chronological split.

q1 = base["date"].quantile(0.50)
q2 = base["date"].quantile(0.75)

base["split"] = np.where(
    base["date"] <= q1,
    "train",
    np.where(base["date"] <= q2, "validation", "test"),
)

split_counts = base.groupby("split").size().reset_index(name="rows")

split_counts.to_csv(
    OUT_DIR / "split_counts.csv",
    index=False,
)

display(split_counts)


In [ ]:
# Cell 5. Evaluate raw probability sources.

source_defs = {
    "market_T24": [
        "market_p_away",
        "market_p_draw",
        "market_p_home",
    ],
    "football_walkforward": [
        "football_p_away",
        "football_p_draw",
        "football_p_home",
    ],
}

if {
    "outcome_model_p_away",
    "outcome_model_p_draw",
    "outcome_model_p_home",
} <= set(base.columns):
    source_defs["outcome_model_10"] = [
        "outcome_model_p_away",
        "outcome_model_p_draw",
        "outcome_model_p_home",
    ]

if {
    "score_p_away",
    "score_p_draw",
    "score_p_home",
} <= set(base.columns):
    source_defs["score_model_10"] = [
        "score_p_away",
        "score_p_draw",
        "score_p_home",
    ]

source_rows = []

for split, part in base.groupby("split"):
    y = part["outcome"].to_numpy()

    for name, cols in source_defs.items():
        tmp = part.dropna(subset=cols).copy()
        if len(tmp) == 0:
            continue

        source_rows.append({
            **evaluate_probs(
                f"{split}_{name}",
                tmp["outcome"].to_numpy(),
                tmp[cols].to_numpy(),
            ),
            "split": split,
            "source": name,
        })

source_metrics = pd.DataFrame(source_rows)

source_metrics.to_csv(
    OUT_DIR / "raw_source_metrics.csv",
    index=False,
)

display(source_metrics.sort_values(["split", "log_loss"]))


In [ ]:
# Cell 6. Grid search: market-anchored blend.

# We build blends:
# p = market + alpha_f * (football - market)
#            + alpha_s * (score - market)
#            + alpha_o * (outcome_model - market)
#
# This keeps market as anchor and only allows small deviations.

train = base[base["split"] == "train"].copy()
validation = base[base["split"] == "validation"].copy()
test = base[base["split"] == "test"].copy()

market_cols = source_defs["market_T24"]
football_cols = source_defs["football_walkforward"]

has_score = "score_model_10" in source_defs
has_outcome_model = "outcome_model_10" in source_defs

score_cols = source_defs.get("score_model_10")
outcome_cols = source_defs.get("outcome_model_10")

def make_blend(df, alpha_f, alpha_s, alpha_o):
    market = df[market_cols].to_numpy()
    football = df[football_cols].to_numpy()

    p = market + alpha_f * (football - market)

    if has_score and score_cols is not None:
        score = df[score_cols].to_numpy()
        score = np.where(np.isnan(score), market, score)
        p = p + alpha_s * (score - market)

    if has_outcome_model and outcome_cols is not None:
        outm = df[outcome_cols].to_numpy()
        outm = np.where(np.isnan(outm), market, outm)
        p = p + alpha_o * (outm - market)

    return normalize_rows(p)

alpha_grid = [-0.25, 0.0, 0.10, 0.20, 0.30, 0.40, 0.60]

blend_rows = []

for alpha_f in alpha_grid:
    for alpha_s in alpha_grid:
        for alpha_o in alpha_grid:
            # Keep total correction bounded.
            if abs(alpha_f) + abs(alpha_s) + abs(alpha_o) > 0.8:
                continue

            val_p = make_blend(
                validation,
                alpha_f=alpha_f,
                alpha_s=alpha_s,
                alpha_o=alpha_o,
            )

            val_row = evaluate_probs(
                "validation_blend",
                validation["outcome"].to_numpy(),
                val_p,
            )

            test_p = make_blend(
                test,
                alpha_f=alpha_f,
                alpha_s=alpha_s,
                alpha_o=alpha_o,
            )

            test_row = evaluate_probs(
                "test_blend",
                test["outcome"].to_numpy(),
                test_p,
            )

            blend_rows.append({
                "alpha_football": alpha_f,
                "alpha_score": alpha_s,
                "alpha_outcome_model": alpha_o,
                "val_accuracy": val_row["accuracy"],
                "val_log_loss": val_row["log_loss"],
                "val_brier": val_row["brier"],
                "test_accuracy": test_row["accuracy"],
                "test_log_loss": test_row["log_loss"],
                "test_brier": test_row["brier"],
            })

blend_grid = pd.DataFrame(blend_rows)

blend_grid = blend_grid.sort_values(
    ["val_log_loss", "test_log_loss"],
    ascending=[True, True],
)

blend_grid.to_csv(
    OUT_DIR / "market_anchored_blend_grid.csv",
    index=False,
)

best_blend = blend_grid.iloc[0].to_dict()

display(blend_grid.head(20))
print("Best blend:", best_blend)


In [ ]:
# Cell 7. Conservative residual correction model.

# This is a second approach:
# features are deltas against market, not raw odds.
# It learns when football/score disagree with market.
# We keep strong regularization and evaluate only on validation/test.

residual_feature_rows = []

def add_delta_features(df):
    out = df.copy()

    for outcome in ["away", "draw", "home"]:
        out[f"delta_football_{outcome}"] = (
            out[f"football_p_{outcome}"]
            - out[f"market_p_{outcome}"]
        )

        if f"score_p_{outcome}" in out.columns:
            out[f"delta_score_{outcome}"] = (
                out[f"score_p_{outcome}"]
                - out[f"market_p_{outcome}"]
            )

        if f"outcome_model_p_{outcome}" in out.columns:
            out[f"delta_outcome_model_{outcome}"] = (
                out[f"outcome_model_p_{outcome}"]
                - out[f"market_p_{outcome}"]
            )

    return out

resid_df = add_delta_features(base)

resid_features = [
    col for col in resid_df.columns
    if col.startswith("delta_")
    and pd.api.types.is_numeric_dtype(resid_df[col])
]

# Add market entropy / uncertainty as a feature.
market_p = resid_df[market_cols].to_numpy()
resid_df["market_entropy"] = -np.sum(
    market_p * np.log(np.clip(market_p, 1e-12, 1.0)),
    axis=1,
)
resid_features.append("market_entropy")

train_r = resid_df[resid_df["split"] == "train"].copy()
val_r = resid_df[resid_df["split"] == "validation"].copy()
test_r = resid_df[resid_df["split"] == "test"].copy()

residual_model_rows = []

for C in [0.01, 0.03, 0.1, 0.3, 1.0]:
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            C=C,
            random_state=SEED,
        )),
    ])

    model.fit(
        train_r[resid_features],
        train_r["outcome"],
    )

    for split_name, part in [
        ("validation", val_r),
        ("test", test_r),
    ]:
        raw = model.predict_proba(part[resid_features])
        proba = np.zeros((len(part), 3))

        for i, cls in enumerate(model.named_steps["model"].classes_):
            proba[:, int(cls)] = raw[:, i]

        # Blend the learned residual classifier with market.
        # The classifier alone is too unstable, so test shrink values.
        market_part = part[market_cols].to_numpy()

        for shrink in [0.10, 0.20, 0.30, 0.40]:
            p = normalize_rows(
                market_part + shrink * (proba - market_part)
            )

            row = evaluate_probs(
                f"residual_C{C}_shrink{shrink}_{split_name}",
                part["outcome"].to_numpy(),
                p,
            )

            row.update({
                "C": C,
                "shrink": shrink,
                "split": split_name,
            })

            residual_model_rows.append(row)

residual_model_metrics = pd.DataFrame(residual_model_rows)

residual_model_metrics.to_csv(
    OUT_DIR / "residual_correction_model_metrics.csv",
    index=False,
)

display(residual_model_metrics.sort_values(["split", "log_loss"]).head(30))
print("residual features:", resid_features)


In [ ]:
# Cell 8. Build final probability from validation-selected model.

# Pick best validation blend.
best_blend_params = {
    "alpha_football": float(best_blend["alpha_football"]),
    "alpha_score": float(best_blend["alpha_score"]),
    "alpha_outcome_model": float(best_blend["alpha_outcome_model"]),
}

base["final_p_away"] = np.nan
base["final_p_draw"] = np.nan
base["final_p_home"] = np.nan

p_all = make_blend(
    base,
    alpha_f=best_blend_params["alpha_football"],
    alpha_s=best_blend_params["alpha_score"],
    alpha_o=best_blend_params["alpha_outcome_model"],
)

base["final_p_away"] = p_all[:, 0]
base["final_p_draw"] = p_all[:, 1]
base["final_p_home"] = p_all[:, 2]

final_metric_rows = []

for split, part in base.groupby("split"):
    final_metric_rows.append(
        {
            **evaluate_probs(
                f"{split}_final_market_anchored_blend",
                part["outcome"].to_numpy(),
                part[["final_p_away", "final_p_draw", "final_p_home"]].to_numpy(),
            ),
            "split": split,
        }
    )

final_metrics = pd.DataFrame(final_metric_rows)

final_metrics.to_csv(
    OUT_DIR / "final_market_anchored_metrics.csv",
    index=False,
)

base.to_csv(
    OUT_DIR / "final_probability_dataset.csv",
    index=False,
)

display(final_metrics)


In [ ]:
# Cell 9. Edge selector from final probability.

selection_rows = []

for _, row in base.iterrows():
    for outcome_name, cls in [
        ("away", 0),
        ("draw", 1),
        ("home", 2),
    ]:
        model_p = row[f"final_p_{outcome_name}"]
        market_p = row[f"market_p_{outcome_name}"]

        odds_24h = row[f"best_{outcome_name}_odds_24h"]
        odds_1h = row.get(f"best_{outcome_name}_odds_1h", np.nan)

        if pd.isna(model_p) or pd.isna(market_p) or pd.isna(odds_24h):
            continue

        selection_rows.append({
            "date": row["date"],
            "split": row["split"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "outcome": int(row["outcome"]),
            "selection": outcome_name,
            "selection_class": cls,
            "model_p": float(model_p),
            "market_p": float(market_p),
            "edge": float(model_p - market_p),
            "best_odds_24h": float(odds_24h),
            "best_odds_1h": float(odds_1h) if pd.notna(odds_1h) else np.nan,
            "model_ev": float(model_p * odds_24h - 1.0),
            "selection_won": int(int(row["outcome"]) == cls),
            "clv_proxy_odds": (
                safe_div(odds_24h, odds_1h) - 1.0
                if pd.notna(odds_1h) and odds_1h > 0
                else np.nan
            ),
        })

selection_df = pd.DataFrame(selection_rows)

selection_df["profit_1usd"] = selection_df.apply(
    profit_from_pick,
    axis=1,
)

selection_df.to_csv(
    OUT_DIR / "final_selection_rows.csv",
    index=False,
)

display(selection_df.head(20))
print("selection rows:", len(selection_df))


In [ ]:
# Cell 10. Edge strategy grid using final probability.

def summarize_picks(picks):
    if len(picks) == 0:
        return {
            "n_picks": 0,
            "hit_rate": np.nan,
            "roi_flat": np.nan,
            "profit_flat": 0.0,
            "avg_clv": np.nan,
            "median_clv": np.nan,
            "avg_edge": np.nan,
            "avg_ev": np.nan,
        }

    return {
        "n_picks": int(len(picks)),
        "hit_rate": float(picks["selection_won"].mean()),
        "roi_flat": float(picks["profit_1usd"].mean()),
        "profit_flat": float(picks["profit_1usd"].sum()),
        "avg_clv": float(picks["clv_proxy_odds"].mean()),
        "median_clv": float(picks["clv_proxy_odds"].median()),
        "avg_edge": float(picks["edge"].mean()),
        "avg_ev": float(picks["model_ev"].mean()),
    }

val_sel = selection_df[selection_df["split"] == "validation"].copy()
test_sel = selection_df[selection_df["split"] == "test"].copy()

grid_rows = []

for min_edge in [0.005, 0.01, 0.02, 0.03, 0.05]:
    for min_ev in [-0.02, 0.0, 0.02, 0.05]:
        for min_odds, max_odds in [
            (1.20, 3.00),
            (1.50, 4.00),
            (1.80, 6.00),
            (2.00, 8.00),
        ]:
            val_picks = val_sel[
                (val_sel["edge"] >= min_edge)
                & (val_sel["model_ev"] >= min_ev)
                & (val_sel["best_odds_24h"] >= min_odds)
                & (val_sel["best_odds_24h"] <= max_odds)
            ].copy()

            if len(val_picks) < 5:
                continue

            test_picks = test_sel[
                (test_sel["edge"] >= min_edge)
                & (test_sel["model_ev"] >= min_ev)
                & (test_sel["best_odds_24h"] >= min_odds)
                & (test_sel["best_odds_24h"] <= max_odds)
            ].copy()

            val_stats = summarize_picks(val_picks)
            test_stats = summarize_picks(test_picks)

            grid_rows.append({
                "min_edge": min_edge,
                "min_ev": min_ev,
                "min_odds": min_odds,
                "max_odds": max_odds,
                **{f"val_{k}": v for k, v in val_stats.items()},
                **{f"test_{k}": v for k, v in test_stats.items()},
            })

edge_grid = pd.DataFrame(grid_rows)

if len(edge_grid) > 0:
    edge_grid = edge_grid.sort_values(
        ["val_roi_flat", "test_avg_clv", "test_roi_flat"],
        ascending=[False, False, False],
    )

edge_grid.to_csv(
    OUT_DIR / "final_edge_strategy_grid.csv",
    index=False,
)

display(edge_grid.head(30))


In [ ]:
# Cell 11. Conservative shortlist and final decision.

if len(edge_grid) == 0:
    shortlist = pd.DataFrame()
else:
    shortlist = edge_grid[
        (edge_grid["val_n_picks"] >= 8)
        & (edge_grid["test_n_picks"] >= 8)
        & (edge_grid["val_roi_flat"] > 0)
        & (edge_grid["test_roi_flat"] > 0)
        & (edge_grid["test_avg_clv"] > -0.02)
    ].copy()

    shortlist = shortlist.sort_values(
        ["test_avg_clv", "test_roi_flat", "val_roi_flat"],
        ascending=[False, False, False],
    )

shortlist.to_csv(
    OUT_DIR / "conservative_edge_shortlist.csv",
    index=False,
)

red_flags = []

if len(shortlist) == 0:
    red_flags.append("no conservative edge strategy survived")

test_final = final_metrics[final_metrics["split"] == "test"]

if len(test_final) > 0:
    # Compare against market test from raw source metrics.
    market_test = source_metrics[
        (source_metrics["split"] == "test")
        & (source_metrics["source"] == "market_T24")
    ]

    if len(market_test) > 0:
        final_ll = float(test_final.iloc[0]["log_loss"])
        market_ll = float(market_test.iloc[0]["log_loss"])

        if final_ll > market_ll:
            red_flags.append("final blend has worse test log loss than market")

decision = {
    "real_money_allowed": False,
    "paper_tracking_allowed": True,
    "best_blend_params": best_blend_params,
    "red_flags": red_flags,
    "recommendation": (
        "Use only for paper tracking. "
        "Require bigger historical odds sample and non-negative CLV "
        "before real-money discussion."
    ),
}

save_json(
    OUT_DIR / "decision_report.json",
    decision,
)

display(shortlist.head(20))
display(pd.DataFrame([decision]))


In [ ]:
# Cell 12. Save report zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "rows_total": int(len(base)),
    "source_count": int(len(source_defs)),
    "best_blend_params": best_blend_params,
    "selection_rows": int(len(selection_df)),
    "edge_grid_rows": int(len(edge_grid)),
    "shortlist_rows": int(len(shortlist)),
    "real_money_allowed": False,
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = PROCESSED_DIR / "11_market_anchored_correction_report_bundle.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
